In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
!pip install biopython --target=$nb_path

In [ ]:
import os
import numpy as np
import pandas as pd
from pandas import Series, DataFrame
import random

import itertools
import math

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import TensorDataset, Dataset, DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
for fold in range(1, 6):
    fold = fold
    base_model_dir = "/content/drive/MyDrive/@antigen_epitope/abdb_v4"
    new_base_model_dir = "/content/drive/MyDrive/@antigen_epitope/abdb_new_method/"
    testing_dir = os.path.join(base_model_dir, f'fold_{fold}/data_test_processed_step1_with_SI/')
    input_file_path = os.path.join(testing_dir, 'cnn_testing_patch_3points.txt')
    df_dir = os.path.join(new_base_model_dir, f'fold_{fold}/data_test/')
    filtered_data_path = os.path.join(df_dir, 'all_data_with_SI.csv')


    protein_names = []
    patch_indices = []
    patch_summaries = []

    with open(input_file_path, 'r') as file:
        lines = file.readlines()
        current_protein_name = None
        for line in lines:
            line = line.strip()
            if line.startswith('Patches for protein'):
                current_protein_name = line.split(' ')[3][:-1]
            elif line.startswith('indices:'):
                indices = line.split('[')[1].split(']')[0].split(', ')
                indices = [int(idx) for idx in indices]
                protein_names.append(current_protein_name)
                patch_indices.append(indices)

    filtered_combined_df = pd.read_csv(filtered_data_path)
    filtered_combined_df = filtered_combined_df[filtered_combined_df['SaSaRate'] >= 0.1]

    for protein_name, indices in zip(protein_names, patch_indices):
        protein_data = filtered_combined_df[filtered_combined_df['name'] == protein_name]

        embeddings = []
        for idx in indices:
            residue_row = protein_data[protein_data['ResID'] == idx]
            if not residue_row.empty:
                embedding = residue_row.iloc[0, -321:-1].values
                embeddings.append(embedding)

        if len(embeddings) == 3:
            patch_embedding_sum = np.sum(embeddings, axis=0)
            patch_summary = {
                'Protein_Name': protein_name,
                'Patch_Indices': indices
            }

            for i, value in enumerate(patch_embedding_sum):
                patch_summary[f'Embedding_{i+1}'] = value

            patch_summaries.append(patch_summary)

    test_df = pd.DataFrame(patch_summaries)
    print(test_df)
    output_dir = os.path.join(new_base_model_dir, f'fold_{fold}/data_test')
    output_file_combined = os.path.join(output_dir, 'test_patch.csv')
    os.makedirs(output_dir, exist_ok=True)
    test_df.to_csv(output_file_combined, index=False)

In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/'
new_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/combined_results/'
os.makedirs(save_dir, exist_ok=True)

all_folds_combined = []

for fold in range(1, 6):
    print(f"Processing fold {fold}...")

    # 1. Load the trained model
    model_dir = os.path.join(new_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'classification_model_with_val.h5')
    model = load_model(model_path)
    print(f"Model for fold {fold} loaded successfully.")

    # 2. Load and process testing data
    testing_dir = os.path.join(new_model_dir, f'fold_{fold}/data_test/')
    testing_data_path = os.path.join(testing_dir, 'test_patch.csv')
    test_df = pd.read_csv(testing_data_path)
    X_test = test_df[[f'Embedding_{i}' for i in range(1, 321)]].to_numpy()

    # 3. Predict and process results
    predictions = model.predict(X_test)
    #binary_predictions = (predictions > 0.5).astype(int)
    test_df['Predicted_Status'] = predictions

    # Save intermediate predictions for debugging (optional)
    output_file_path = os.path.join(save_dir, f'fold_{fold}_predictions.csv')
    test_df.to_csv(output_file_path, index=False)
    print(f"Predictions for fold {fold} saved to {output_file_path}.")

    # 4. Load original testing data (all_data_with_SI.csv)
    data_test_dir = os.path.join(new_model_dir, f'fold_{fold}/data_test/')
    original_data_path = os.path.join(data_test_dir, 'all_data_with_SI.csv')
    original_data_df = pd.read_csv(original_data_path)
    original_data_df = original_data_df[original_data_df['SaSaRate'] >= 0.1]

    # 5. Merge predictions back to the original data
    # Expand 'Patch_Indices' in test_df to long format
    test_df = test_df.assign(Patch_Indices=test_df['Patch_Indices'].str.strip('[]').str.split(', ')).explode('Patch_Indices')
    test_df['Patch_Indices'] = test_df['Patch_Indices'].astype(int)

    # Merge by matching 'name' and 'ResID' with 'Protein_Name' and 'Patch_Indices'
    merged_df = pd.merge(
        original_data_df,
        test_df,
        left_on=['name', 'ResID'],
        right_on=['Protein_Name', 'Patch_Indices'],
        how='left'
    )

    # Fill missing predictions
    merged_df['Predicted_Status'] = pd.to_numeric(merged_df['Predicted_Status'], errors='coerce')

    df_groupby_cols = merged_df.columns.tolist()[:10]

    merged_df = (
        merged_df
        .sort_values(by=['Predicted_Status'], ascending=False)
        .groupby(df_groupby_cols, as_index=False)
        .first()
    )


    # Normalize predictions within each protein group
    def normalize_group(x):
        if x.isnull().all() or (x.max() == x.min()):
            return np.nan
        else:
            return (x - x.min()) / (x.max() - x.min())

    merged_df['Normalized_Predicted_Status'] = merged_df.groupby('name')['Predicted_Status'].transform(normalize_group)

    # Save fold result and append to combined DataFrame
    fold_output_path = os.path.join(save_dir, f'fold_{fold}_final_combined.csv')
    merged_df.to_csv(fold_output_path, index=False)
    print(f"Saved fold {fold} results to {fold_output_path}.")

    all_folds_combined.append(merged_df)

# 6. Combine all folds and save
combined_df = pd.concat(all_folds_combined, ignore_index=True)
combined_output_file = os.path.join(save_dir, 'all_folds_combined.csv')
combined_df.to_csv(combined_output_file, index=False)
print(f"All fold results combined and saved to {combined_output_file}.")

In [ ]:
combined_df

In [ ]:
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/combined_results/'
combined_output_file = os.path.join(save_dir, 'all_folds_combined.csv')
combined_df = pd.read_csv(combined_output_file)
combined_df = combined_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['Normalized_Predicted_Status', 'Predicted_Status', 'Status'])

In [ ]:
combined_df = combined_df[combined_df['SaSaRate'] >= 0.1]

In [ ]:
combined_df

In [ ]:
combined_df['Status']

In [ ]:
combined_df = combined_df.dropna(subset=['Normalized_Predicted_Status'])
combined_df = combined_df.dropna(subset=['Predicted_Status'])
combined_df = combined_df.dropna(subset=['Status'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set up the matplotlib figure
plt.figure(figsize=(10, 6))

# Plot the distribution of pred values for each Status
sns.histplot(data=combined_df, x='Normalized_Predicted_Status', hue='Status', element='step', stat="density", common_norm=False)

# Provide labels and title
plt.xlabel('Prediction Value')
plt.ylabel('Density')
plt.title('Distribution of Prediction Values by Status')
plt.legend(title='Status')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Get true binary labels
true_labels = np.where(combined_df['Status'] == 'Interacting', 1, 0)

# Get prediction scores
pred_scores = combined_df[combined_df['SaSaRate'] >= 0.1]['Normalized_Predicted_Status'].values

# Compute ROC curve and AUC
fpr, tpr, _ = roc_curve(true_labels, pred_scores)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) for PLM')
plt.legend(loc="lower right")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt

y_true = combined_df['Status'].replace({'Interacting': 1, 'Non-Interacting': 0}).astype(int)
y_pred = (combined_df['Predicted_Status'] >= 0.5).astype(int)
y_scores = combined_df['Normalized_Predicted_Status']

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_true, y_scores)

print("Classification Report:")
print(classification_report(y_true, y_pred))
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")

conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 6))
plt.imshow(conf_matrix, cmap="Blues", interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], ['Non-Interacting', 'Interacting'])
plt.yticks([0, 1], ['Non-Interacting', 'Interacting'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

for i in range(2):
    for j in range(2):
        plt.text(j, i, conf_matrix[i, j], ha="center", va="center", color="red")

plt.show()

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Guess')
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.legend()
plt.grid()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc, balanced_accuracy_score, matthews_corrcoef, precision_recall_curve

# Compute ROC curve and AUC
fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)
print(f'ROC-AUC: {roc_auc:.2f}')

# Compute Balanced Accuracy (BAC)
y_scores = (combined_df['Normalized_Predicted_Status'] > 0.5).astype(int)
bac = balanced_accuracy_score(y_true, y_scores)
print(f'BAC: {bac:.2f}')

# Compute Matthews Correlation Coefficient (MCC)
mcc = matthews_corrcoef(y_true, y_scores)
print(f'MCC: {mcc:.2f}')

# Compute Precision-Recall curve and AUC
precision, recall, _ = precision_recall_curve(y_true, y_scores)
pr_auc = auc(recall, precision)
print(f'PR-AUC: {pr_auc:.2f}')

# Compute Nantigens (number of unique antigens)
nantigens = combined_df['name'].nunique()
print(f'Nantigens: {nantigens}')

# Compute Fpredicted (mean fraction of surface residues predicted as epitopes)
f_predicted = np.mean(y_scores) * 100
print(f'Fpredicted: {f_predicted:.2f}%')

In [ ]:
print(combined_df['Status'].value_counts())

In [ ]:
# Calculate AUC for each protein
protein_names = combined_df['name'].unique()
auc_list = []

for protein in protein_names:
    protein_df = combined_df[combined_df['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        auc = roc_auc_score(true_labels, protein_df['Normalized_Predicted_Status'])
        auc_list.append(auc)

# Plot AUC distribution
plt.hist(auc_list, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('AUC')
plt.ylabel('Frequency')
plt.title('Distribution of AUC for each protein without large patch prediction')
plt.grid(True)
plt.show()

print('The mean AUC of protein is', np.mean(auc_list))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, recall_score

# Calculate recall for each protein
protein_names = combined_df['name'].unique()
recall_list = []

for protein in protein_names:
    protein_df = combined_df[combined_df['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        fpr, tpr, thresholds = roc_curve(true_labels, protein_df['Normalized_Predicted_Status'])
        J = tpr - fpr
        idx = np.argmax(J)
        best_threshold = thresholds[idx]
        recall = recall_score(true_labels, (protein_df['Normalized_Predicted_Status'] >= best_threshold).astype(int))
        recall_list.append(recall)

# Plot recall distribution
plt.hist(recall_list, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('Recall')
plt.ylabel('Frequency')
plt.title('Distribution of recall for each protein without large patch prediction')
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, accuracy_score

# Calculate accuracy for each protein
protein_names = combined_df['name'].unique()
accuracy_list = []

for protein in protein_names:
    protein_df = combined_df[combined_df['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        fpr, tpr, thresholds = roc_curve(true_labels, protein_df['Normalized_Predicted_Status'])
        J = tpr - fpr
        idx = np.argmax(J)
        best_threshold = thresholds[idx]
        accuracy = accuracy_score(true_labels, (protein_df['Normalized_Predicted_Status'] >= best_threshold).astype(int))
        accuracy_list.append(accuracy)

# Plot accuracy distribution
plt.hist(accuracy_list, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('Accuracy')
plt.ylabel('Frequency')
plt.title('Distribution of accuracy for each protein without large patch prediction')
plt.grid(True)
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import torch
from torch.utils.data import DataLoader
from sklearn.cluster import DBSCAN

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/'
new_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/combined_results/'
os.makedirs(save_dir, exist_ok=True)

all_folds_combined = []

for fold in range(1, 6):
    print(f"Processing fold {fold}...")

    # 1. Load the trained model
    model_dir = os.path.join(new_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'classification_model_with_val.h5')
    model = load_model(model_path)
    print(f"Model for fold {fold} loaded successfully.")

    # 2. Load and process testing data
    testing_dir = os.path.join(new_model_dir, f'fold_{fold}/data_test/')
    testing_data_path = os.path.join(testing_dir, 'test_patch.csv')
    test_df = pd.read_csv(testing_data_path)
    X_test = test_df[[f'Embedding_{i}' for i in range(1, 321)]].to_numpy()

    # 3. Predict and process results
    predictions = model.predict(X_test)
    #binary_predictions = (predictions > 0.5).astype(int)
    test_df['Predicted_Status'] = predictions

    # Save intermediate predictions for debugging (optional)
    output_file_path = os.path.join(save_dir, f'fold_{fold}_predictions.csv')
    test_df.to_csv(output_file_path, index=False)
    print(f"Predictions for fold {fold} saved to {output_file_path}.")

    # 4. Load original testing data (all_data_with_SI.csv)
    data_test_dir = os.path.join(new_model_dir, f'fold_{fold}/data_test/')
    original_data_path = os.path.join(data_test_dir, 'all_data_with_SI.csv')
    original_data_df = pd.read_csv(original_data_path)
    original_data_df = original_data_df[original_data_df['SaSaRate'] >= 0.1]

    # 5. Merge predictions back to the original data
    # Expand 'Patch_Indices' in test_df to long format
    test_df = test_df.assign(Patch_Indices=test_df['Patch_Indices'].str.strip('[]').str.split(', ')).explode('Patch_Indices')
    test_df['Patch_Indices'] = test_df['Patch_Indices'].astype(int)

    # Merge by matching 'name' and 'ResID' with 'Protein_Name' and 'Patch_Indices'
    merged_df = pd.merge(
        original_data_df,
        test_df,
        left_on=['name', 'ResID'],
        right_on=['Protein_Name', 'Patch_Indices'],
        how='left'
    )

    # Fill missing predictions
    merged_df['Predicted_Status'] = pd.to_numeric(merged_df['Predicted_Status'], errors='coerce')

    df_groupby_cols = merged_df.columns.tolist()[:10]

    merged_df = (
        merged_df
        .sort_values(by=['Predicted_Status'], ascending=False)
        .groupby(df_groupby_cols, as_index=False)
        .first()
    )


    # Normalize predictions within each protein group
    def normalize_group(x):
        if x.isnull().all() or (x.max() == x.min()):
            return np.nan
        else:
            return (x - x.min()) / (x.max() - x.min())

    merged_df['Normalized_Predicted_Status'] = merged_df.groupby('name')['Predicted_Status'].transform(normalize_group)

    def top_clusters_with_highest_pred(group,distance_threshold,num_clusters):
        # Extract coordinates
        coords = group[['Cax', 'Cay', 'Caz']].values

        # Use DBSCAN to cluster residues based on the distance threshold
        clustering = DBSCAN(eps=distance_threshold, min_samples=1).fit(coords)
        group['cluster'] = clustering.labels_

        # Calculate the prediction sum for each cluster
        cluster_pred_sums = group.groupby('cluster')['Normalized_Predicted_Status'].max().sort_values(ascending=False)

        # Select the top clusters based on the sum of predictions
        top_clusters = cluster_pred_sums.head(num_clusters).index

        # Set 'pred' to 1 for residues in the top clusters, 0 otherwise
        group['Normalized_Predicted_Status'] = group['cluster'].apply(lambda x: 1 if x in top_clusters else 0)

        # Drop the 'cluster' column
        group = group.drop(columns=['cluster'])

        return group

    distance_threshold = 4.5
    num_clusters = 3

    # Apply the function to each group in the DataFrame
    merged_df = merged_df.groupby('name').apply(lambda x: top_clusters_with_highest_pred(x,distance_threshold,num_clusters)).reset_index(drop=True)

    # Save fold result and append to combined DataFrame
    fold_output_path = os.path.join(save_dir, f'fold_{fold}_final_combined.csv')
    merged_df.to_csv(fold_output_path, index=False)
    print(f"Saved fold {fold} results to {fold_output_path}.")

    all_folds_combined.append(merged_df)

# 6. Combine all folds and save
combined_df = pd.concat(all_folds_combined, ignore_index=True)
combined_output_file = os.path.join(save_dir, 'all_folds_combined_using_large_patch.csv')
combined_df.to_csv(combined_output_file, index=False)
print(f"All fold results combined and saved to {combined_output_file}.")

In [ ]:
merged_df

In [ ]:
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/combined_results/'
combined_output_file = os.path.join(save_dir, 'all_folds_combined_using_large_patch.csv')
combined_df = pd.read_csv(combined_output_file)
combined_df = combined_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['Normalized_Predicted_Status', 'Predicted_Status', 'Status'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set up the matplotlib figure
plt.figure(figsize=(10, 6))

# Plot the distribution of pred values for each Status
sns.histplot(data=combined_df, x='Normalized_Predicted_Status', hue='Status', element='step', stat="density", common_norm=False)

# Provide labels and title
plt.xlabel('Prediction Value')
plt.ylabel('Density')
plt.title('Distribution of Prediction Values by using large patch')
plt.legend(title='Status')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Get true binary labels
true_labels = np.where(combined_df['Status'] == 'Interacting', 1, 0)

# Get prediction scores
pred_scores = combined_df[combined_df['SaSaRate'] >= 0.1]['Normalized_Predicted_Status'].values

# Compute ROC curve and AUC
fpr, tpr, _ = roc_curve(true_labels, pred_scores)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) by using large patch')
plt.legend(loc="lower right")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt

y_true = combined_df['Status'].replace({'Interacting': 1, 'Non-Interacting': 0}).astype(int)
y_pred = (combined_df['Predicted_Status'] >= 0.5).astype(int)
y_scores = combined_df['Normalized_Predicted_Status']

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_true, y_scores)

print("Classification Report:")
print(classification_report(y_true, y_pred))
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")

conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 6))
plt.imshow(conf_matrix, cmap="Blues", interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], ['Non-Interacting', 'Interacting'])
plt.yticks([0, 1], ['Non-Interacting', 'Interacting'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

for i in range(2):
    for j in range(2):
        plt.text(j, i, conf_matrix[i, j], ha="center", va="center", color="red")

plt.show()

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Guess')
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc, balanced_accuracy_score, matthews_corrcoef, precision_recall_curve

# Compute ROC curve and AUC
fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)
print(f'ROC-AUC: {roc_auc:.2f}')

# Compute Balanced Accuracy (BAC)
bac = balanced_accuracy_score(y_true, y_scores)
print(f'BAC: {bac:.2f}')

# Compute Matthews Correlation Coefficient (MCC)
mcc = matthews_corrcoef(y_true, y_scores)
print(f'MCC: {mcc:.2f}')

# Compute Precision-Recall curve and AUC
precision, recall, _ = precision_recall_curve(y_true, y_scores)
pr_auc = auc(recall, precision)
print(f'PR-AUC: {pr_auc:.2f}')

# Compute Nantigens (number of unique antigens)
nantigens = combined_df['name'].nunique()
print(f'Nantigens: {nantigens}')

# Compute Fpredicted (mean fraction of surface residues predicted as epitopes)
f_predicted = np.mean(y_scores) * 100
print(f'Fpredicted: {f_predicted:.2f}%')

In [ ]:
# Calculate AUC for each protein
protein_names = combined_df['name'].unique()
auc_list = []

for protein in protein_names:
    protein_df = combined_df[combined_df['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        auc = roc_auc_score(true_labels, protein_df['Normalized_Predicted_Status'])
        auc_list.append(auc)

# Plot AUC distribution
plt.hist(auc_list, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('AUC')
plt.ylabel('Frequency')
plt.title('Distribution of AUC for each protein without large patch prediction')
plt.grid(True)
plt.show()

print('The mean AUC of protein is', np.mean(auc_list))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, recall_score

# Calculate recall for each protein
protein_names = combined_df['name'].unique()
recall_list = []

for protein in protein_names:
    protein_df = combined_df[combined_df['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        fpr, tpr, thresholds = roc_curve(true_labels, protein_df['Normalized_Predicted_Status'])
        J = tpr - fpr
        idx = np.argmax(J)
        best_threshold = thresholds[idx]
        recall = recall_score(true_labels, (protein_df['Normalized_Predicted_Status'] >= best_threshold).astype(int))
        recall_list.append(recall)

# Plot recall distribution
plt.hist(recall_list, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('Recall')
plt.ylabel('Frequency')
plt.title('Distribution of recall for each protein without large patch prediction')
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, accuracy_score

# Calculate accuracy for each protein
protein_names = combined_df['name'].unique()
accuracy_list = []

for protein in protein_names:
    protein_df = combined_df[combined_df['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        fpr, tpr, thresholds = roc_curve(true_labels, protein_df['Normalized_Predicted_Status'])
        J = tpr - fpr
        idx = np.argmax(J)
        best_threshold = thresholds[idx]
        accuracy = accuracy_score(true_labels, (protein_df['Normalized_Predicted_Status'] >= best_threshold).astype(int))
        accuracy_list.append(accuracy)

# Plot accuracy distribution
plt.hist(accuracy_list, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('Accuracy')
plt.ylabel('Frequency')
plt.title('Distribution of accuracy for each protein without large patch prediction')
plt.grid(True)
plt.show()

In [ ]:
from tensorflow.keras.models import load_model
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/'
new_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/combined_results/'

all_folds_combined = []

for fold in range(1, 6):
    print(f"Processing fold {fold}...")

    # 1. Load the trained model
    model_dir = os.path.join(new_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'classification_model_with_val.h5')
    model = load_model(model_path)
    print(f"Model for fold {fold} loaded successfully.")

    data_test_dir = os.path.join(new_model_dir, f'fold_{fold}/data_test/')
    original_data_path = os.path.join(data_test_dir, 'all_data_with_SI.csv')
    original_data_df = pd.read_csv(original_data_path)
    merged_df = original_data_df[original_data_df['SaSaRate'] >= 0.1]
    X_test = merged_df[[f'embedding_{i}' for i in range(0, 320)]].to_numpy()
    predictions = model.predict(X_test)
    merged_df['Predicted_Status'] = predictions

    # Fill missing predictions
    merged_df['Predicted_Status'] = pd.to_numeric(merged_df['Predicted_Status'], errors='coerce')

    df_groupby_cols = merged_df.columns.tolist()[:10]

    merged_df = (
        merged_df
        .sort_values(by=['Predicted_Status'], ascending=False)
        .groupby(df_groupby_cols, as_index=False)
        .first()
    )


    # Normalize predictions within each protein group
    def normalize_group(x):
        if x.isnull().all() or (x.max() == x.min()):
            return np.nan
        else:
            return (x - x.min()) / (x.max() - x.min())

    merged_df['Normalized_Predicted_Status'] = merged_df.groupby('name')['Predicted_Status'].transform(normalize_group)

    # Save fold result and append to combined DataFrame
    fold_output_path = os.path.join(save_dir, f'fold_{fold}_final_combined.csv')
    merged_df.to_csv(fold_output_path, index=False)
    print(f"Saved fold {fold} results to {fold_output_path}.")

    all_folds_combined.append(merged_df)

# 6. Combine all folds and save
combined_df = pd.concat(all_folds_combined, ignore_index=True)
combined_output_file = os.path.join(save_dir, 'all_folds_combined_directly_pred.csv')
combined_df.to_csv(combined_output_file, index=False)
print(f"All fold results combined and saved to {combined_output_file}.")

In [ ]:
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/combined_results/'
combined_output_file = os.path.join(save_dir, 'all_folds_combined_directly_pred.csv')
combined_df = pd.read_csv(combined_output_file)
combined_df = combined_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['Normalized_Predicted_Status', 'Predicted_Status', 'Status'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set up the matplotlib figure
plt.figure(figsize=(10, 6))

# Plot the distribution of pred values for each Status
sns.histplot(data=combined_df, x='Normalized_Predicted_Status', hue='Status', element='step', stat="density", common_norm=False)

# Provide labels and title
plt.xlabel('Prediction Value')
plt.ylabel('Density')
plt.title('Distribution of Prediction Values by Status')
plt.legend(title='Status')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Get true binary labels
true_labels = np.where(combined_df['Status'] == 'Interacting', 1, 0)

# Get prediction scores
pred_scores = combined_df[combined_df['SaSaRate'] >= 0.1]['Normalized_Predicted_Status'].values

# Compute ROC curve and AUC
fpr, tpr, _ = roc_curve(true_labels, pred_scores)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt

y_true = combined_df['Status'].replace({'Interacting': 1, 'Non-Interacting': 0}).astype(int)
y_pred = (combined_df['Predicted_Status'] >= 0.5).astype(int)
y_scores = combined_df['Normalized_Predicted_Status']

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_true, y_scores)

print("Classification Report:")
print(classification_report(y_true, y_pred))
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")

conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 6))
plt.imshow(conf_matrix, cmap="Blues", interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], ['Non-Interacting', 'Interacting'])
plt.yticks([0, 1], ['Non-Interacting', 'Interacting'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

for i in range(2):
    for j in range(2):
        plt.text(j, i, conf_matrix[i, j], ha="center", va="center", color="red")

plt.show()

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Guess')
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Calculate AUC for each protein
protein_names = combined_df['name'].unique()
auc_list = []

for protein in protein_names:
    protein_df = combined_df[combined_df['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        auc = roc_auc_score(true_labels, protein_df['Normalized_Predicted_Status'])
        auc_list.append(auc)

# Plot AUC distribution
plt.hist(auc_list, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('AUC')
plt.ylabel('Frequency')
plt.title('Distribution of AUC for each protein without large patch prediction')
plt.grid(True)
plt.show()

print('The mean AUC of protein is', np.mean(auc_list))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, recall_score

# Calculate recall for each protein
protein_names = combined_df['name'].unique()
recall_list = []

for protein in protein_names:
    protein_df = combined_df[combined_df['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        fpr, tpr, thresholds = roc_curve(true_labels, protein_df['Normalized_Predicted_Status'])
        J = tpr - fpr
        idx = np.argmax(J)
        best_threshold = thresholds[idx]
        recall = recall_score(true_labels, (protein_df['Normalized_Predicted_Status'] >= best_threshold).astype(int))
        recall_list.append(recall)

# Plot recall distribution
plt.hist(recall_list, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('Recall')
plt.ylabel('Frequency')
plt.title('Distribution of recall for each protein without large patch prediction')
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, accuracy_score

# Calculate accuracy for each protein
protein_names = combined_df['name'].unique()
accuracy_list = []

for protein in protein_names:
    protein_df = combined_df[combined_df['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        fpr, tpr, thresholds = roc_curve(true_labels, protein_df['Normalized_Predicted_Status'])
        J = tpr - fpr
        idx = np.argmax(J)
        best_threshold = thresholds[idx]
        accuracy = accuracy_score(true_labels, (protein_df['Normalized_Predicted_Status'] >= best_threshold).astype(int))
        accuracy_list.append(accuracy)

# Plot accuracy distribution
plt.hist(accuracy_list, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('Accuracy')
plt.ylabel('Frequency')
plt.title('Distribution of accuracy for each protein without large patch prediction')
plt.grid(True)
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import torch
from torch.utils.data import DataLoader
from sklearn.cluster import DBSCAN

# Paths
new_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/combined_results/'
os.makedirs(save_dir, exist_ok=True)

all_folds_combined = []

for fold in range(1, 6):
    print(f"Processing fold {fold}...")

    # 1. Load the trained model
    model_dir = os.path.join(new_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'classification_model_with_val.h5')
    model = load_model(model_path)
    print(f"Model for fold {fold} loaded successfully.")

    data_test_dir = os.path.join(new_model_dir, f'fold_{fold}/data_test/')
    original_data_path = os.path.join(data_test_dir, 'all_data_with_SI.csv')
    original_data_df = pd.read_csv(original_data_path)
    merged_df = original_data_df[original_data_df['SaSaRate'] >= 0.1]
    X_test = merged_df[[f'embedding_{i}' for i in range(0, 320)]].to_numpy()
    predictions = model.predict(X_test)
    merged_df['Predicted_Status'] = predictions

    # Fill missing predictions
    merged_df['Predicted_Status'] = pd.to_numeric(merged_df['Predicted_Status'], errors='coerce')

    df_groupby_cols = merged_df.columns.tolist()[:10]

    merged_df = (
        merged_df
        .sort_values(by=['Predicted_Status'], ascending=False)
        .groupby(df_groupby_cols, as_index=False)
        .first()
    )


    # Normalize predictions within each protein group
    def normalize_group(x):
        if x.isnull().all() or (x.max() == x.min()):
            return np.nan
        else:
            return (x - x.min()) / (x.max() - x.min())

    merged_df['Normalized_Predicted_Status'] = merged_df.groupby('name')['Predicted_Status'].transform(normalize_group)

    def top_clusters_with_highest_pred(group,distance_threshold,num_clusters):
        # Extract coordinates
        coords = group[['Cax', 'Cay', 'Caz']].values

        # Use DBSCAN to cluster residues based on the distance threshold
        clustering = DBSCAN(eps=distance_threshold, min_samples=1).fit(coords)
        group['cluster'] = clustering.labels_

        # Calculate the prediction sum for each cluster
        cluster_pred_sums = group.groupby('cluster')['Normalized_Predicted_Status'].max().sort_values(ascending=False)

        # Select the top clusters based on the sum of predictions
        top_clusters = cluster_pred_sums.head(num_clusters).index

        # Set 'pred' to 1 for residues in the top clusters, 0 otherwise
        group['Normalized_Predicted_Status'] = group['cluster'].apply(lambda x: 1 if x in top_clusters else 0)

        # Drop the 'cluster' column
        group = group.drop(columns=['cluster'])

        return group

    distance_threshold = 4.5
    num_clusters = 3

    # Apply the function to each group in the DataFrame
    merged_df = merged_df.groupby('name').apply(lambda x: top_clusters_with_highest_pred(x,distance_threshold,num_clusters)).reset_index(drop=True)

    # Save fold result and append to combined DataFrame
    fold_output_path = os.path.join(save_dir, f'fold_{fold}_final_combined.csv')
    merged_df.to_csv(fold_output_path, index=False)
    print(f"Saved fold {fold} results to {fold_output_path}.")

    all_folds_combined.append(merged_df)

# 6. Combine all folds and save
combined_df = pd.concat(all_folds_combined, ignore_index=True)
combined_output_file = os.path.join(save_dir, 'all_folds_combined_using_large_patch_directly_pred.csv')
combined_df.to_csv(combined_output_file, index=False)
print(f"All fold results combined and saved to {combined_output_file}.")

In [ ]:
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/combined_results/'
combined_output_file = os.path.join(save_dir, 'all_folds_combined_using_large_patch_directly_pred.csv')
combined_df = pd.read_csv(combined_output_file)
combined_df = combined_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['Normalized_Predicted_Status', 'Predicted_Status', 'Status'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set up the matplotlib figure
plt.figure(figsize=(10, 6))

# Plot the distribution of pred values for each Status
sns.histplot(data=combined_df, x='Normalized_Predicted_Status', hue='Status', element='step', stat="density", common_norm=False)

# Provide labels and title
plt.xlabel('Prediction Value')
plt.ylabel('Density')
plt.title('Distribution of Prediction Values by Status')
plt.legend(title='Status')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Get true binary labels
true_labels = np.where(combined_df['Status'] == 'Interacting', 1, 0)

# Get prediction scores
pred_scores = combined_df[combined_df['SaSaRate'] >= 0.1]['Normalized_Predicted_Status'].values

# Compute ROC curve and AUC
fpr, tpr, _ = roc_curve(true_labels, pred_scores)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt

y_true = combined_df['Status'].replace({'Interacting': 1, 'Non-Interacting': 0}).astype(int)
y_pred = (combined_df['Predicted_Status'] >= 0.5).astype(int)
y_scores = combined_df['Normalized_Predicted_Status']

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_true, y_scores)

print("Classification Report:")
print(classification_report(y_true, y_pred))
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")

conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 6))
plt.imshow(conf_matrix, cmap="Blues", interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], ['Non-Interacting', 'Interacting'])
plt.yticks([0, 1], ['Non-Interacting', 'Interacting'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

for i in range(2):
    for j in range(2):
        plt.text(j, i, conf_matrix[i, j], ha="center", va="center", color="red")

plt.show()

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Guess')
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.legend()
plt.grid()
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, recall_score, accuracy_score

protein_names = combined_df['name'].unique()

for protein in protein_names:
    protein_df = combined_df[combined_df['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        fpr, tpr, thresholds = roc_curve(true_labels, protein_df['Normalized_Predicted_Status'])
        J = tpr - fpr
        idx = np.argmax(J)
        best_threshold = thresholds[idx]
        accuracy = accuracy_score(true_labels, (protein_df['Normalized_Predicted_Status'] >= best_threshold).astype(int))
        if accuracy > 0.95:
          print(protein)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
protein_names = combined_df['name'].unique()

for protein in protein_names:
    protein_df = combined_df[combined_df['name'] == protein]
    true_labels = np.where(protein_df['Status'] == 'Interacting', 1, 0)
    if len(np.unique(true_labels)) == 2:
        auc = roc_auc_score(true_labels, protein_df['Normalized_Predicted_Status'])
        if auc < 0.4:
          print(protein)

In [ ]:
import os
import pandas as pd
from Bio.PDB import PDBParser, PDBIO, is_aa

best_names = [
"6N81_2_L_H_A"
]
best_df = combined_df[combined_df['name'].isin(best_names)]
best_df['ResID'] = best_df['ResID'].astype(str)

input_pdb_path = "/content/drive/MyDrive/@antigen_epitope/abdb_v3/data/structures/"
output_pdb_path = "/content/drive/MyDrive/@antigen_epitope/abdb_new_method/combined_results/bad/"

parser = PDBParser(QUIET=True)
io = PDBIO()

unique_names = best_df['name'].unique()

for pdb_name in unique_names:
    pdb_filename = pdb_name + '.pdb'

    input_file = None
    input_file = input_pdb_path + pdb_filename

    structure = parser.get_structure(pdb_name, input_file)

    df_subset = best_df[best_df['name'] == pdb_name].copy()
    status_dict = dict(zip(df_subset['ResID'], df_subset['Normalized_Predicted_Status']))

    for model in structure:
        for chain in model:
            for residue in chain:
                if not is_aa(residue, standard=True):
                    continue

                res_id_tuple = residue.get_id()  # e.g. (' ', 123, ' ')
                res_id = f"{res_id_tuple[0]}{res_id_tuple[1]}{res_id_tuple[2]}".strip()

                if chain.id in ['L', 'H']:
                    pred_value = 0.0
                else:
                    pred_value = status_dict.get(res_id, 0.0)

                for atom in residue.get_atoms():
                    if pred_value > 0.9:
                      pred_value = 1.0
                    else:
                      pred_value = 0.0
                    atom.set_bfactor(pred_value)

    output_file = os.path.join(output_pdb_path, pdb_filename)
    io.set_structure(structure)
    io.save(output_file)
    print(f"已处理并保存：{output_file}")

```pymol
spectrum b, rainbow, minimum=0, maximum=1
color magenta, chain H or chain L
set internal_gui, off
```


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tensorflow.keras.models import load_model

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/'
new_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/combined_results/'
os.makedirs(save_dir, exist_ok=True)

for fold in range(1, 6):
    print(f"Processing fold {fold}...")

    # Load the trained model
    model_dir = os.path.join(new_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'classification_model_with_val.h5')
    model = load_model(model_path)
    print(f"Model for fold {fold} loaded successfully.")

    # Load and process testing data
    testing_dir = os.path.join(new_model_dir, f'fold_{fold}/data_test/')
    testing_data_path = os.path.join(testing_dir, 'test_patch.csv')
    test_df = pd.read_csv(testing_data_path)
    X_test = test_df[[f'Embedding_{i}' for i in range(1, 321)]].to_numpy()

    # Predict and process results
    predictions = model.predict(X_test).flatten()
    test_df['Predicted_Status'] = predictions
    output_file_path = os.path.join(save_dir, f'fold_{fold}_predictions.csv')
    test_df.to_csv(output_file_path, index=False)

    # Load original testing data (all_data_with_SI.csv)
    original_data_path = os.path.join(testing_dir, 'all_data_with_SI.csv')
    original_data_df = pd.read_csv(original_data_path)
    original_data_df = original_data_df[original_data_df['SaSaRate'] >= 0.1]

    # Expand Patch_Indices and merge with original data
    test_df = test_df.assign(Patch_Indices=test_df['Patch_Indices'].str.strip('[]').str.split(', ')).explode('Patch_Indices')
    test_df['Patch_Indices'] = test_df['Patch_Indices'].astype(int)
    merged_df = pd.merge(original_data_df, test_df, left_on=['name', 'ResID'], right_on=['Protein_Name', 'Patch_Indices'], how='left')

    # Fill missing predictions and normalize
    merged_df['Predicted_Status'] = pd.to_numeric(merged_df['Predicted_Status'], errors='coerce')
    merged_df['Normalized_Predicted_Status'] = merged_df.groupby('name')['Predicted_Status'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))

    # Save results
    fold_output_path = os.path.join(save_dir, f'fold_{fold}_final_combined.csv')
    merged_df.to_csv(fold_output_path, index=False)

    # === EVALUATION ===
    merged_df = merged_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['Normalized_Predicted_Status', 'Predicted_Status', 'Status'])
    print(f"Evaluating model performance for fold {fold}...")
    true_labels = (merged_df['Status'] == 'Interacting').astype(int).values
    pred_scores = merged_df['Normalized_Predicted_Status'].values

    # Compute ROC curve and AUC
    fpr, tpr, thresholds = roc_curve(true_labels, pred_scores)
    roc_auc = auc(fpr, tpr)

    # Compute Youden's J statistic and best threshold
    J = tpr - fpr
    idx = np.argmax(J)
    best_threshold = thresholds[idx]

    # Compute classification metrics
    binary_predictions = (pred_scores >= best_threshold).astype(int)
    accuracy = accuracy_score(true_labels, binary_predictions)
    precision = precision_score(true_labels, binary_predictions)
    recall = recall_score(true_labels, binary_predictions)
    f1 = f1_score(true_labels, binary_predictions)
    roc_auc_value = roc_auc_score(true_labels, pred_scores)

    # Print metrics
    print(f"Fold {fold} Metrics:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC: {roc_auc_value:.4f}")
    print(f"Best Cutoff (Threshold): {best_threshold:.3f}")

    # Plot ROC curve
    plt.figure(figsize=(8, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f}')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - Fold {fold}')
    plt.legend(loc="lower right")
    plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tensorflow.keras.models import load_model

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/'
new_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/combined_results/'
os.makedirs(save_dir, exist_ok=True)

for fold in range(1, 6):
    print(f"Processing fold {fold}...")

    # 1. Load the trained model
    model_dir = os.path.join(new_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'classification_model_with_val.h5')
    model = load_model(model_path)
    print(f"Model for fold {fold} loaded successfully.")

    data_test_dir = os.path.join(new_model_dir, f'fold_{fold}/data_test/')
    original_data_path = os.path.join(data_test_dir, 'all_data_with_SI.csv')
    original_data_df = pd.read_csv(original_data_path)
    merged_df = original_data_df[original_data_df['SaSaRate'] >= 0.1]
    X_test = merged_df[[f'embedding_{i}' for i in range(0, 320)]].to_numpy()
    predictions = model.predict(X_test)
    merged_df['Predicted_Status'] = predictions

    # Fill missing predictions
    merged_df['Predicted_Status'] = pd.to_numeric(merged_df['Predicted_Status'], errors='coerce')

    df_groupby_cols = merged_df.columns.tolist()[:10]

    merged_df = (
        merged_df
        .sort_values(by=['Predicted_Status'], ascending=False)
        .groupby(df_groupby_cols, as_index=False)
        .first()
    )


    # Normalize predictions within each protein group
    def normalize_group(x):
        if x.isnull().all() or (x.max() == x.min()):
            return np.nan
        else:
            return (x - x.min()) / (x.max() - x.min())

    merged_df['Normalized_Predicted_Status'] = merged_df.groupby('name')['Predicted_Status'].transform(normalize_group)

    # Save fold result and append to combined DataFrame
    fold_output_path = os.path.join(save_dir, f'fold_{fold}_final_combined.csv')
    merged_df.to_csv(fold_output_path, index=False)

    # === EVALUATION ===
    merged_df = merged_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['Normalized_Predicted_Status', 'Predicted_Status', 'Status'])
    print(f"Evaluating model performance for fold {fold}...")
    true_labels = (merged_df['Status'] == 'Interacting').astype(int).values
    pred_scores = merged_df['Normalized_Predicted_Status'].values

    # Compute ROC curve and AUC
    fpr, tpr, thresholds = roc_curve(true_labels, pred_scores)
    roc_auc = auc(fpr, tpr)

    # Compute Youden's J statistic and best threshold
    J = tpr - fpr
    idx = np.argmax(J)
    best_threshold = thresholds[idx]

    # Compute classification metrics
    binary_predictions = (pred_scores >= best_threshold).astype(int)
    accuracy = accuracy_score(true_labels, binary_predictions)
    precision = precision_score(true_labels, binary_predictions)
    recall = recall_score(true_labels, binary_predictions)
    f1 = f1_score(true_labels, binary_predictions)
    roc_auc_value = roc_auc_score(true_labels, pred_scores)

    # Print metrics
    print(f"Fold {fold} Metrics:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC: {roc_auc_value:.4f}")
    print(f"Best Cutoff (Threshold): {best_threshold:.3f}")

    # Plot ROC curve
    plt.figure(figsize=(8, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f}')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - Fold {fold}')
    plt.legend(loc="lower right")
    plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, balanced_accuracy_score, matthews_corrcoef, precision_recall_curve
from tensorflow.keras.models import load_model

# Paths
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4/'
new_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/combined_results/'
os.makedirs(save_dir, exist_ok=True)

for fold in range(1, 6):
    print(f"Processing fold {fold}...")

    # 1. Load the trained model
    model_dir = os.path.join(new_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'classification_model_with_val.h5')
    model = load_model(model_path)
    print(f"Model for fold {fold} loaded successfully.")

    data_test_dir = os.path.join(new_model_dir, f'fold_{fold}/data_test/')
    original_data_path = os.path.join(data_test_dir, 'all_data_with_SI.csv')
    original_data_df = pd.read_csv(original_data_path)
    merged_df = original_data_df[original_data_df['SaSaRate'] >= 0.1]
    X_test = merged_df[[f'embedding_{i}' for i in range(0, 320)]].to_numpy()
    predictions = model.predict(X_test)
    merged_df['Predicted_Status'] = predictions

    # Fill missing predictions
    merged_df['Predicted_Status'] = pd.to_numeric(merged_df['Predicted_Status'], errors='coerce')

    df_groupby_cols = merged_df.columns.tolist()[:10]

    merged_df = (
        merged_df
        .sort_values(by=['Predicted_Status'], ascending=False)
        .groupby(df_groupby_cols, as_index=False)
        .first()
    )


    # Normalize predictions within each protein group
    def normalize_group(x):
        if x.isnull().all() or (x.max() == x.min()):
            return np.nan
        else:
            return (x - x.min()) / (x.max() - x.min())

    merged_df['Normalized_Predicted_Status'] = merged_df.groupby('name')['Predicted_Status'].transform(normalize_group)

    # Save fold result and append to combined DataFrame
    fold_output_path = os.path.join(save_dir, f'fold_{fold}_final_combined.csv')
    merged_df.to_csv(fold_output_path, index=False)

    # === EVALUATION ===
    merged_df = merged_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['Normalized_Predicted_Status', 'Predicted_Status', 'Status'])
    print(f"Evaluating model performance for fold {fold}...")
    true_labels = (merged_df['Status'] == 'Interacting').astype(int).values
    pred_scores = merged_df['Normalized_Predicted_Status'].values

    # Compute ROC curve and AUC
    fpr, tpr, thresholds = roc_curve(true_labels, pred_scores)
    roc_auc = auc(fpr, tpr)

    # Compute Youden's J statistic and best threshold
    J = tpr - fpr
    idx = np.argmax(J)
    best_threshold = thresholds[idx]

    # Compute classification metrics
    binary_predictions = (pred_scores >= best_threshold).astype(int)
    accuracy = accuracy_score(true_labels, binary_predictions)
    precision = precision_score(true_labels, binary_predictions)
    recall = recall_score(true_labels, binary_predictions)
    f1 = f1_score(true_labels, binary_predictions)
    roc_auc_value = roc_auc_score(true_labels, pred_scores)

    # Print metrics
    print(f"Fold {fold} Metrics:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC: {roc_auc_value:.4f}")
    print(f"Best Cutoff (Threshold): {best_threshold:.3f}")
    # Compute ROC curve and AUC
    roc_auc = auc(fpr, tpr)
    print(f'ROC-AUC: {roc_auc:.2f}')

    y_true = merged_df['Status'].replace({'Interacting': 1, 'Non-Interacting': 0}).astype(int)
    # Compute Balanced Accuracy (BAC)
    y_scores = (merged_df['Normalized_Predicted_Status'] > 0.5).astype(int)
    bac = balanced_accuracy_score(y_true, y_scores)
    print(f'BAC: {bac:.2f}')

    # Compute Matthews Correlation Coefficient (MCC)
    mcc = matthews_corrcoef(y_true, y_scores)
    print(f'MCC: {mcc:.2f}')

    # Compute Precision-Recall curve and AUC
    precision_vals, recall_vals, _ = precision_recall_curve(y_true, pred_scores)
    pr_auc = auc(recall_vals, precision_vals)
    print(f'PR-AUC: {pr_auc:.2f}')

    # Compute Nantigens (number of unique antigens)
    nantigens = merged_df['name'].nunique()
    print(f'Nantigens: {nantigens}')

    # Compute Fpredicted (mean fraction of surface residues predicted as epitopes)
    f_predicted = np.mean(y_scores) * 100
    print(f'Fpredicted: {f_predicted:.2f}%')

    # Plot ROC curve
    plt.figure(figsize=(8, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f}')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - Fold {fold}')
    plt.legend(loc="lower right")
    plt.show()

